# AIRPRED — Stage 2: Model Training (Colab)

Trains all four variants (A, B, C/AIRPRED, D) and saves their checkpoints to
Drive. **Evaluation, metrics, and statistical tests are NOT in this notebook**
— see `AIRPRED_Stage3_Colab.ipynb` for that, which loads these checkpoints
back up in a separate session. Keeping training and evaluation as separate
notebooks means you can re-run evaluation later (different metric breakdowns,
rechecking a specific city) without retraining anything.

**Before running:** make sure Stage 1 finished successfully and its sanity
check passed (mean derived PM2.5 was NOT flagged as implausible, and the
validation window count recovered to ~20,000, not the ~950 from before the
BLH fix).

**Enable the free GPU first:** Runtime → Change runtime type → T4 GPU → Save.

## 0. Confirm GPU is on, get the AIRPRED code onto this runtime

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — go to Runtime > Change runtime type > T4 GPU, then re-run this cell.")
    print("(Training will still work on CPU, just much slower.)")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/AIRPRED'   # match the path used in the Stage 1 notebook
import os
print("Contents of data/final (from Stage 1):", os.listdir(f'{PROJECT_DIR}/data/final'))

In [ ]:
import sys, os, shutil

REPO_URL = "https://github.com/<your-username>/AIRPRED.git"  # <-- set this to your actual repo URL, same as Stage 1

# Same private-repo note as Stage 1: plain HTTPS clone fails on a private
# repo. Either make it public, or use a Personal Access Token via Colab's
# secrets manager -- see Stage 1 notebook's Cell 1 comments for the exact code.

if os.path.exists('/content/AIRPRED'):
    shutil.rmtree('/content/AIRPRED')  # clean re-clone, avoids errors on reruns

!git clone {REPO_URL} /content/AIRPRED

sys.path.insert(0, '/content/AIRPRED')
os.chdir('/content/AIRPRED')

import numpy as np
import pandas as pd
import yaml
import joblib
import torch

from src.models.variants import (
    VariantA_SingleBranchUnified, VariantB_DualBranchConcat,
    VariantC_AIRPRED, VariantD_PM25Only,
)
from src.training.train import train_variant, set_seed
print("AIRPRED source imported successfully.")

## 1. Load Stage 1 output and config

Loads the `train.npz` / `val.npz` / `test.npz` / `scaler.pkl` that Stage 1 saved to
Drive. If you ran Stage 1 in a different session without Drive, re-upload them here
instead with `files.upload()`.

In [ ]:
cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['training']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
print(yaml.dump(cfg))

In [ ]:
DATA_DIR = f'{PROJECT_DIR}/data/final'

def load_split(name):
    d = np.load(f'{DATA_DIR}/{name}.npz', allow_pickle=True)
    return d['X_pm25'], d['X_met'], d['Y'], d['cities']

train_data = load_split('train')[:3]   # (X_pm25, X_met, Y) -- train_variant doesn't need city labels
val_data = load_split('val')[:3]

for name, d in [('train', train_data), ('val', val_data)]:
    print(name, [a.shape for a in d])
print("\nCheck: val sample count should be ~18,000-21,000 (post BLH-fix), not ~950.")

## 2. Train all four variants — identical hyperparameters, per config.yaml

Runs sequentially: A, B, C (AIRPRED), D. Each saves its own best-validation-RMSE
checkpoint to Drive. This is the step that actually benefits from the GPU — expect
each variant to finish in well under the CPU estimate from before.

In [ ]:
VARIANTS = {
    "A": VariantA_SingleBranchUnified,
    "B": VariantB_DualBranchConcat,
    "C": VariantC_AIRPRED,
    "D": VariantD_PM25Only,
}

CKPT_DIR = f'{PROJECT_DIR}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

horizon = cfg['data']['forecast_horizon']
trained_models = {}
best_val_rmses = {}

for name, ModelClass in VARIANTS.items():
    print(f"\n{'='*60}\nTraining Variant {name}\n{'='*60}")
    set_seed(cfg['training']['random_seed'])  # BEFORE model construction -- controls weight init
    model = ModelClass(horizon=horizon) if name != "C" else ModelClass(
        horizon=horizon, d_model=cfg['model']['d_model'], num_heads=cfg['model']['num_attention_heads']
    )
    ckpt_path = f'{CKPT_DIR}/variant_{name.lower()}_best.pt'
    best_rmse = train_variant(model, train_data, val_data, cfg, ckpt_path)
    model.load_state_dict(torch.load(ckpt_path))  # restore best checkpoint, not just last epoch
    trained_models[name] = model
    best_val_rmses[name] = best_rmse
    print(f"Variant {name} best val RMSE (scaled units): {best_rmse:.4f}")

print("\nAll four variants trained.")
print(best_val_rmses)

## Done

All four checkpoints are saved to `Drive/AIRPRED/checkpoints/variant_<x>_best.pt`.
Open `AIRPRED_Stage3_Colab.ipynb` next — it loads these checkpoints and the
Stage 1 test set to actually produce metrics, tables, and statistical tests.